### Load processed data

In [65]:
import geopandas as gpd
from pathlib import Path

fire_data_by_countries = gpd.read_file(
    Path.cwd().parent.joinpath("data/processed", "fire_data_by_countries.gpkg")
)
fire_data_by_countries_dates = gpd.read_file(
    Path.cwd().parent.joinpath("data/processed", "fire_data_by_countries_dates.gpkg")
)
countries_boundaries = gpd.read_file(
    Path.cwd().parent.joinpath("data/raw", "geoboundaries_world.geojson")
).to_crs("EPSG:4326")

fire_data_by_countries.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   iso_code      194 non-null    str     
 1   name          194 non-null    str     
 2   country_area  194 non-null    float64 
 3   fire_area     194 non-null    float64 
 4   area_perc     194 non-null    float64 
 5   geometry      194 non-null    geometry
dtypes: float64(3), geometry(1), str(2)
memory usage: 9.2 KB


### Simplify polygons
Using the non-simplified polygons created the resulting html map was huge and taking a performance hit. Unlike `simplify()`, `simplify_coverage()` assumes that the GeoSeries forms a polygonal coverage. Polygons borders remain thus consistent.

In [70]:
fire_data_by_countries_simplified = fire_data_by_countries.copy()
fire_data_by_countries_simplified["geometry"] = (
    fire_data_by_countries.geometry.simplify_coverage(tolerance=0.05)
)
countries_boundaries_simplified = countries_boundaries.copy()
countries_boundaries_simplified["geometry"] = (
    countries_boundaries.geometry.simplify_coverage(tolerance=0.05)
)

### Create `output/` directory if not present yet

In [73]:
dir_path = Path(Path.cwd().parent.joinpath("output"))
if not dir_path.exists():
    Path.mkdir(dir_path)

In [ ]:
import folium

map = folium.Map(tiles="CartoDB Positron")

folium.Choropleth(
    geo_data=countries_boundaries_simplified,
    name="Percentage of wildfire area",
    data=fire_data_by_countries,
    columns=[
        "name",
        "area_perc",
    ],
    key_on="feature.properties.shapeName",
    fill_color="viridis",
    fill_opacity=0.6,
    line_opacity=0.2,
    legend_name="Area affected by wildfires (%)",
).add_to(map)

folium.GeoJson(
    fire_data_by_countries_simplified,
    name="Interactive Tooltips",
    style_function=lambda x: {"fillColor": "#ffffff00", "color": "#ffffff00"},
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "area_perc"],
        aliases=["Country:", "Percentage of area affected:"],
        localize=True,
    ),
).add_to(map)

map.save(Path.cwd().parent.joinpath("output", "map.html"))
map